# Full Pipeline Ablation Study

This notebook evaluates the contribution of each major component in the proposed image translation pipeline.

1. Full pipeline:
Stable Diffusion + BLIP-2 + MiDaS + ControlNet + IP-Adapter + text prompt + reference image

2. Without IP-Adapter:
removes image-based guidance

3. Without ControlNet:
removes structural depth map guidance

4. Without BLIP-2:
removes text description generation

5. Baseline:
Stable Diffusion only

In [ ]:
!pip -q install -r /content/image-data-generation/requirements.txt

In [ ]:
import os
import time
import gc
import torch
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt

from transformers import Blip2Processor, Blip2ForConditionalGeneration
from diffusers import (
    StableDiffusionImg2ImgPipeline,
    ControlNetModel,
    StableDiffusionControlNetImg2ImgPipeline
)

In [ ]:
dataset_dir = "/content/image-data-generation/dataset/source_road_scenes"
reference_image_path = "/content/image-data-generation/dataset/weather_reference_conditions/snow_reference_1.jpg"
results_dir = "/content/image-data-generation/results/full_pipeline_ablation"

os.makedirs(results_dir, exist_ok=True)

image_paths = sorted([
    os.path.join(dataset_dir, f)
    for f in os.listdir(dataset_dir)
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".jfif"))
])

if len(image_paths) == 0:
    raise FileNotFoundError("No road-scene images found in the dataset folder.")

print(f"Found {len(image_paths)} road-scene images.")


In [ ]:
user_prompt = "Convert image to snowy winter with heavy snowfall"

if not os.path.exists(reference_image_path):
    raise FileNotFoundError(f"Reference image not found: {reference_image_path}")

reference_image = Image.open(reference_image_path).convert("RGB").resize((512, 512))

negative_prompt = (
    "blurry, distorted cars, distorted road, bad geometry, "
    "low quality, warped perspective, artifacts"
)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print("Device:", device)

In [ ]:
print("Loading BLIP-2...")

processor = Blip2Processor.from_pretrained(
    "Salesforce/blip2-opt-2.7b"
)

blip_model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    torch_dtype=torch.float16,
    device_map="auto"
)


captions = {}

for image_path in image_paths:
    image_id = os.path.splitext(os.path.basename(image_path))[0]
    image = Image.open(image_path).convert("RGB")

    inputs = processor(image, return_tensors="pt").to(blip_model.device)

    with torch.no_grad():
        ids = blip_model.generate(**inputs, max_length=40)

    caption = processor.decode(ids[0], skip_special_tokens=True)
    captions[image_id] = caption

    print(image_id, ":", caption)

del blip_model
del processor
gc.collect()
torch.cuda.empty_cache()


In [ ]:
print("Loading MiDaS...")

midas_device = torch.device(device)

midas_model = torch.hub.load("intel-isl/MiDaS", "DPT_Hybrid")
midas_model.to(midas_device).eval()

midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
transform = midas_transforms.dpt_transform

In [ ]:
print("Loading ControlNet...")

controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-depth",
    torch_dtype=dtype
)

In [ ]:
print("Loading Stable Diffusion + ControlNet + IP-Adapter...")

full_pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
    "sd-legacy/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=dtype,
    safety_checker=None
).to(device)

full_pipe.load_ip_adapter(
    "h94/IP-Adapter",
    subfolder="models",
    weight_name="ip-adapter_sd15.bin"
)

full_pipe.set_ip_adapter_scale(0.3)

In [ ]:
print("Loading baseline Stable Diffusion only pipeline...")

baseline_pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    "sd-legacy/stable-diffusion-v1-5",
    torch_dtype=dtype,
    safety_checker=None
).to(device)

print("Loading Stable Diffusion + IP-Adapter pipeline...")

ip_pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    "sd-legacy/stable-diffusion-v1-5",
    torch_dtype=dtype,
    safety_checker=None
).to(device)

ip_pipe.load_ip_adapter(
    "h94/IP-Adapter",
    subfolder="models",
    weight_name="ip-adapter_sd15.bin"
)

ip_pipe.set_ip_adapter_scale(0.3)

In [ ]:
for image_path in image_paths:
    image_id = os.path.splitext(os.path.basename(image_path))[0]
    output_dir = os.path.join(results_dir, image_id)
    os.makedirs(output_dir, exist_ok=True)

    print("\nProcessing:", image_id)

    image = Image.open(image_path).convert("RGB")
    init_image = image.resize((512, 512))

    init_image.save(os.path.join(output_dir, "original.png"))
    reference_image.save(os.path.join(output_dir, "reference_condition.png"))

    manual_prompt = (
        f"{user_prompt}. "
        "Preserve original road layout, vehicles, object positions, and perspective."
    )

    blip_prompt = (
        f"Scene description: {captions[image_id]}. "
        f"User prompt: {user_prompt}. "
        "Preserve original road layout, vehicles, object positions, and perspective."
    )

    # MiDaS depth
    img_np = np.array(image)
    input_batch = transform(img_np).to(midas_device)

    with torch.no_grad():
        prediction = midas_model(input_batch)
        prediction = torch.nn.functional.interpolate(
            prediction.unsqueeze(1),
            size=img_np.shape[:2],
            mode="bicubic",
            align_corners=False
        ).squeeze(0).squeeze(0)

    depth = prediction.detach().cpu().numpy()
    depth_vis = cv2.normalize(depth, None, 0, 255, norm_type=cv2.NORM_MINMAX)
    depth_vis = depth_vis.astype(np.uint8)

    depth_gray = Image.fromarray(depth_vis).convert("RGB").resize((512, 512))
    depth_gray.save(os.path.join(output_dir, "depth_gray.png"))

    # 1. Full pipeline
    full_pipe.set_ip_adapter_scale(0.3)
    generator = torch.Generator(device=device).manual_seed(42)

    start = time.time()
    full_result = full_pipe(
        prompt=blip_prompt,
        negative_prompt=negative_prompt,
        image=init_image,
        control_image=depth_gray,
        ip_adapter_image=reference_image,
        strength=0.8,
        guidance_scale=7.5,
        controlnet_conditioning_scale=1.0,
        num_inference_steps=30,
        generator=generator
    ).images[0]
    full_time = time.time() - start
    full_result.save(os.path.join(output_dir, "full_pipeline.png"))

    # 2. Without IP-Adapter
    full_pipe.set_ip_adapter_scale(0.0)
    generator = torch.Generator(device=device).manual_seed(42)

    start = time.time()
    no_ip_result = full_pipe(
        prompt=blip_prompt,
        negative_prompt=negative_prompt,
        image=init_image,
        control_image=depth_gray,
        ip_adapter_image=reference_image,
        strength=0.8,
        guidance_scale=7.5,
        controlnet_conditioning_scale=1.0,
        num_inference_steps=30,
        generator=generator
    ).images[0]
    no_ip_time = time.time() - start
    no_ip_result.save(os.path.join(output_dir, "without_ipadapter.png"))

    # 3. Without BLIP-2
    full_pipe.set_ip_adapter_scale(0.3)
    generator = torch.Generator(device=device).manual_seed(42)

    start = time.time()
    no_blip_result = full_pipe(
        prompt=manual_prompt,
        negative_prompt=negative_prompt,
        image=init_image,
        control_image=depth_gray,
        ip_adapter_image=reference_image,
        strength=0.8,
        guidance_scale=7.5,
        controlnet_conditioning_scale=1.0,
        num_inference_steps=30,
        generator=generator
    ).images[0]
    no_blip_time = time.time() - start
    no_blip_result.save(os.path.join(output_dir, "without_blip2.png"))

    # 4. Baseline Stable Diffusion only
    generator = torch.Generator(device=device).manual_seed(42)

    start = time.time()
    baseline_result = baseline_pipe(
        prompt=manual_prompt,
        negative_prompt=negative_prompt,
        image=init_image,
        strength=0.8,
        guidance_scale=7.5,
        num_inference_steps=30,
        generator=generator
    ).images[0]
    baseline_time = time.time() - start
    baseline_result.save(os.path.join(output_dir, "baseline_sd.png"))

    # 5. Without ControlNet
    ip_pipe.set_ip_adapter_scale(0.3)
    generator = torch.Generator(device=device).manual_seed(42)

    start = time.time()
    no_controlnet_result = ip_pipe(
        prompt=blip_prompt,
        negative_prompt=negative_prompt,
        image=init_image,
        ip_adapter_image=reference_image,
        strength=0.8,
        guidance_scale=7.5,
        num_inference_steps=30,
        generator=generator
    ).images[0]

    no_controlnet_time = time.time() - start
    no_controlnet_result.save(os.path.join(output_dir, "without_controlnet.png"))

    fig, axes = plt.subplots(1, 7, figsize=(28, 5))

    outputs = [
        ("Original", init_image),
        ("Reference", reference_image),
        ("Baseline SD", baseline_result),
        ("No ControlNet", no_controlnet_result),
        ("No IP-Adapter", no_ip_result),
        ("No BLIP-2", no_blip_result),
        ("Full Pipeline", full_result),
    ]

    for ax, (title, img) in zip(axes, outputs):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "comparison.png"), dpi=300)
    plt.show()
    plt.close()

    # Metadata
    with open(os.path.join(output_dir, "metadata.txt"), "w") as f:
        f.write(f"Image ID: {image_id}\n")
        f.write(f"Input path: {image_path}\n")
        f.write(f"Reference image: {reference_image_path}\n")
        f.write(f"BLIP-2 caption: {captions[image_id]}\n")
        f.write(f"Manual prompt: {manual_prompt}\n")
        f.write(f"BLIP prompt: {blip_prompt}\n")
        f.write(f"Negative prompt: {negative_prompt}\n")
        f.write("Seed: 42\n")
        f.write("Image size: 512x512\n")
        f.write("Strength: 0.8\n")
        f.write("Guidance scale: 7.5\n")
        f.write("Inference steps: 30\n")
        f.write("ControlNet conditioning scale: 1.0\n")
        f.write("Full pipeline IP-Adapter scale: 0.3\n")
        f.write("Without IP-Adapter scale: 0.0\n")
        f.write(f"Full pipeline runtime: {full_time:.2f} seconds\n")
        f.write(f"Without IP-Adapter runtime: {no_ip_time:.2f} seconds\n")
        f.write(f"Without BLIP-2 runtime: {no_blip_time:.2f} seconds\n")
        f.write(f"Without ControlNet runtime: {no_controlnet_time:.2f} seconds\n")
        f.write(f"Baseline runtime: {baseline_time:.2f} seconds\n")

    print("Saved results to:", output_dir)

    expected_files = [
        "original.png",
        "reference_condition.png",
        "depth_gray.png",
        "baseline_sd.png",
        "without_controlnet.png",
        "without_ipadapter.png",
        "without_blip2.png",
        "full_pipeline.png",
        "comparison.png",
        "metadata.txt"
    ]

    missing_files = [
        file for file in expected_files
        if not os.path.exists(os.path.join(output_dir, file))
    ]

    if missing_files:
        print("Missing files:", missing_files)
    else:
        print("All expected files saved for", image_id)

    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
del baseline_pipe
del full_pipe
del ip_pipe
del controlnet
del midas_model
del midas_transforms
del transform

gc.collect()
torch.cuda.empty_cache()

print("GPU memory cleared.")